In [64]:
!sudo apt update
!sudo apt intstall ffmpeg
!pip install --upgrade pip
!pip install --quiet torch transformers accelerate ffmpeg ffmpeg-python jiwer bert_score torch bleurt-pytorch protobuf pydub

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Reading package lists... Done                        
Building dependency tree... Done
Reading state information... Done
60 packages can be upgraded. Run 'apt list --upgradable' to see them.
E: Invalid operation intstall


In [66]:
from pydub import AudioSegment

def convert_m4a_to_mp3(input_file, output_file):
    audio = AudioSegment.from_file(input_file, format="m4a")
    audio.export(output_file, format="mp3")

# Example usage
#convert_m4a_to_mp3("input.m4a", "output.mp3")

In [61]:
import torch
from transformers import pipeline

pipeline = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-base"
)

# Change this to a local file - mp3, flac, whatever
result = pipeline("mlk.flac")

print(result["text"])

Device set to use cpu
/opt/conda/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


 I have a dream that one day this nation will rise up and live out the true meaning of its creed.


| Model Size        | Parameters | Relative Speed ⏱️ | Relative Accuracy 🎯 | Notes |
|-------------------|------------|-------------------|-----------------------|-------|
| `tiny`            | ~39M       | 🔥 Fastest         | 😬 Lowest              | Good for simple tasks or quick tests. |
| `base`            | ~74M       | ⚡ Very Fast        | 🤏 Low-Mid             | Decent general-purpose model. |
| `small`           | ~244M      | 🚀 Fast             | 👍 Moderate            | Better performance on diverse speech. |
| `medium`          | ~769M      | 🐎 Slower           | ✅ High                | More accurate with varied accents and noise. |
| `large`           | ~1.55B     | 🐢 Slowest          | 🥇 Highest             | Best transcription quality; supports many languages. |

# Exercise: create your own recordings 
## Windows users can use Sound Recorder
## Mac user can use Voice Memo

Experiment with using your different accents to see if it can detect accurately or if it's biased towards cerrtain voices?
_______________________________________

`jiwer` (short for **"Just Another Word Error Rate"**) is a **Python library** that calculates common **ASR evaluation metrics** like:

- **WER** – Word Error Rate  
- **CER** – Character Error Rate  
- **MER** – Match Error Rate  
- **WIL** – Word Information Lost  

It compares a **reference transcription** (usually the ground truth or manual transcription) with a **hypothesis** (the ASR output) and computes how many insertions, deletions, and substitutions were made.


| **Metric** | **What it Measures** | **When to Use It** | **What It Tells You** | **Example** | **Evaluator Focus** |
|------------|----------------------|---------------------|------------------------|-------------|----------------------|
| **WER** (Word Error Rate) | Incorrect words (insertions, deletions, substitutions) | Most common; word-level comparison | Measures transcription accuracy at the word level | Ref: *"I have a dream today"*<br>Hyp: *"I had a dreams today.”* | Are the **words** correct and in the right position? |
| **CER** (Character Error Rate) | Incorrect characters | Short texts, spelling-sensitive cases | Measures fine-grained spelling and character accuracy | Ref: *"cat"*<br>Hyp: *"kat"* | Are **letters** spelled correctly? Useful for typos or names. |
| **MER** (Match Error Rate) | Reference words not matched (ignores extra insertions) | When extra info isn't critical | How much of the intended message was correctly captured | Ref: *"Turn off the light."*<br>Hyp: *"Please turn off the light now."* | Did the model **understand the core message**, even with fluff? |
| **WIL** (Word Information Lost) | Info lost to the listener, even if some words are correct | When “understandability” is key | More nuanced view of how usable the text is | Ref: *"Take the left exit."*<br>Hyp: *"Take the next exit."* | Would a listener **misunderstand** the result? Focus on meaning. |

In [58]:
from bert_score import score

ref = [
    "I have a dream that one day this nation will rise up", 
    "There is a bird in the sky"
]
hyp = [
    "I had a dream one day the nation will rise",
    "I hurt my foot"
]

P, R, F1 = score(hyp, ref, lang="en", verbose=True)

# Print individual F1 scores
for i, f1 in enumerate(F1):
    print(f"Pair {i+1} - BERTScore F1: {f1:.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 3.27 seconds, 0.61 sentences/sec
Pair 1 - BERTScore F1: 0.9530
Pair 2 - BERTScore F1: 0.8669


In [27]:
from jiwer import wer, cer, mer, wil

reference = "I have a dream that one day this nation will rise up"
hypothesis = "I had a dream one day the nation will rise"

print("WER:", wer(reference, hypothesis))
print("CER:", cer(reference, hypothesis))
print("MER:", mer(reference, hypothesis))
print("WIL:", wil(reference, hypothesis))


WER: 0.75
CER: 0.6666666666666666
MER: 0.5
WIL: 0.625


In [59]:
import torch
from bleurt_pytorch import BleurtConfig, BleurtForSequenceClassification, BleurtTokenizer

config = BleurtConfig.from_pretrained('lucadiliello/BLEURT-20-D12')
model = BleurtForSequenceClassification.from_pretrained('lucadiliello/BLEURT-20-D12')
tokenizer = BleurtTokenizer.from_pretrained('lucadiliello/BLEURT-20-D12')

references = [
    "I have a dream that one day this nation will rise up", 
    "I'm drying my socks"
]
candidates = [
    "I had a dream one day the nation will rise", 
    "Look, there's a spanner"
]

model.eval()
with torch.no_grad():
    inputs = tokenizer(references, candidates, padding='longest', return_tensors='pt')
    res = model(**inputs).logits.flatten().tolist()
for i, r in enumerate(res):
    print(f"Pair {i+1} - Similarity Score: {r:.4f}")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BleurtSPTokenizer'. 
The class this function is called from is 'BertTokenizer'.


Pair 1 - Similarity Score: 0.7018
Pair 2 - Similarity Score: 0.1222


# Exercises
#### 1. Evaluate your own speech-to-text results using the above methods
#### 2. How can they be plotted in meaningful ways?
#### 3. Plot results comparing the different ways in which asr can be evaluated. Add more comparisons to give better results if necessary.
#### 4. What insights can you draw?

# Discussion questions
#### We can evaluate the text generate by ASR via character error rate or word error rate, but what about the latter phonetic/phonological features?

#### What else is missing from ASR generated text?

#### Does ASR create language data of interest to linguists? Which linguists?